# SHIFA TAMEER-E-MILLAT UNIVERSITY
## Shifa School of Computing | BSAI-6
### Course: Natural Language Processing
**Student:** Muhammad Jawad  
**Assignment:** #2  
**Topic:** NLP Text Processing on a Real PDF Document

---
## Step 0: Install & Import All Required Libraries

In [1]:
!pip install pypdf nltk scikit-learn plotly pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 19.0 MB/s eta 0:00:00


In [2]:
import re
import string
import warnings
warnings.filterwarnings('ignore')

from pypdf import PdfReader

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ All libraries imported successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✅ All libraries imported successfully!


---
## Q1(a): PDF Reading and Text Extraction

In [3]:

PDF_PATH = "book.pdf"
reader = PdfReader(PDF_PATH)

total_pages = len(reader.pages)
print(f"📄 Total Number of Pages in PDF: {total_pages}")
print("-" * 50)

📄 Total Number of Pages in PDF: 375
--------------------------------------------------


In [4]:
print("Extracting text from all pages...")

all_text_pages = []
full_text = ""

for i, page in enumerate(reader.pages):
    page_text = page.extract_text() or ""
    all_text_pages.append(page_text)
    full_text += " " + page_text

print(f"✅ Text extracted from {total_pages} pages.")
print(f"   Total characters extracted : {len(full_text):,}")
print(f"   Total words (raw)          : {len(full_text.split()):,}")

Extracting text from all pages...
✅ Text extracted from 375 pages.
   Total characters extracted : 786,555
   Total words (raw)          : 135,503


In [5]:
print("=" * 60)
print("SAMPLE EXTRACTED TEXT (First 1000 characters):")
print("=" * 60)
print(full_text[:1000])
print("\n" + "=" * 60)
print("SAMPLE EXTRACTED TEXT (Page 5):")
print("=" * 60)
print(all_text_pages[4][:500] if len(all_text_pages) > 4 else "N/A")

SAMPLE EXTRACTED TEXT (First 1000 characters):
 Natural Language Processing in Python Authors: Steven Bird, Ewan Klein, Edward Loper
V ersion: 0.9.1 (draft only, please send feedback to authors)
Copyright: © 2001-2008 the authors
License: Creative Commons Attribution-Noncommercial-No Derivative Works 3.0 United
States License
Revision:
Date:
January 24, 2008 2 Bird, Klein & Loper Contents
1 Introduction to Natural Language Processing 19
1.1 The Language Challenge . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19
1.1.1 The Richness of Language . . . . . . . . . . . . . . . . . . . . . . . . . . . . 19
1.1.2 The Promise of NLP . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 20
1.2 Language and Computation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 21
1.2.1 NLP and Intelligence . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 21
1.2.2 Language and Symbol Processing . . . . . . . . . . . . . . . . . . . . . . . . 22
1.2.3 P

---
## Q1(b): Text Preprocessing

Applying all required preprocessing steps with **Regex** patterns clearly shown.

In [6]:
text_lower = full_text.lower()
print("STEP 1 — Lowercase Conversion")
print(f"  Sample: {text_lower[:200]}")
print()

STEP 1 — Lowercase Conversion
  Sample:  natural language processing in python authors: steven bird, ewan klein, edward loper
v ersion: 0.9.1 (draft only, please send feedback to authors)
copyright: © 2001-2008 the authors
license: creative



In [7]:
REGEX_NUMBERS = r'\d+'
text_no_numbers = re.sub(REGEX_NUMBERS, '', text_lower)

print("STEP 2 — Remove Numbers")
print(f"  Regex Pattern : {REGEX_NUMBERS!r}")
print(f"  Sample: {text_no_numbers[:200]}")
print()

STEP 2 — Remove Numbers
  Regex Pattern : '\\d+'
  Sample:  natural language processing in python authors: steven bird, ewan klein, edward loper
v ersion: .. (draft only, please send feedback to authors)
copyright: © - the authors
license: creative commons at



In [8]:
REGEX_SPECIAL = r'[^a-z\s]'
text_no_special = re.sub(REGEX_SPECIAL, '', text_no_numbers)

print("STEP 3 — Remove Special Symbols")
print(f"  Regex Pattern : {REGEX_SPECIAL!r}")
print(f"  Sample: {text_no_special[:200]}")
print()

STEP 3 — Remove Special Symbols
  Regex Pattern : '[^a-z\\s]'
  Sample:  natural language processing in python authors steven bird ewan klein edward loper
v ersion  draft only please send feedback to authors
copyright   the authors
license creative commons attributionnonc



In [9]:
REGEX_SPACES = r'\s+'
text_clean_spaces = re.sub(REGEX_SPACES, ' ', text_no_special).strip()

print("STEP 4 — Remove Extra Spaces")
print(f"  Regex Pattern : {REGEX_SPACES!r}")
print(f"  Sample: {text_clean_spaces[:200]}")
print()

STEP 4 — Remove Extra Spaces
  Regex Pattern : '\\s+'
  Sample: natural language processing in python authors steven bird ewan klein edward loper v ersion draft only please send feedback to authors copyright the authors license creative commons attributionnoncomme



In [10]:
REGEX_PUNCT = r'[' + re.escape(string.punctuation) + r']'
text_no_punct = re.sub(REGEX_PUNCT, '', text_clean_spaces)

print("STEP 5 — Remove Punctuation")
print(f"  Regex Pattern : {REGEX_PUNCT!r}")
print(f"  Sample: {text_no_punct[:200]}")
print()

STEP 5 — Remove Punctuation
  Regex Pattern : '[!"\\#\\$%\\&\'\\(\\)\\*\\+,\\-\\./:;<=>\\?@\\[\\\\\\]\\^_`\\{\\|\\}\\~]'
  Sample: natural language processing in python authors steven bird ewan klein edward loper v ersion draft only please send feedback to authors copyright the authors license creative commons attributionnoncomme



In [11]:
tokens = word_tokenize(text_no_punct)

print("STEP 6 — Tokenization")
print(f"  Total tokens : {len(tokens):,}")
print(f"  First 30 tokens: {tokens[:30]}")
print()

STEP 6 — Tokenization
  Total tokens : 113,118
  First 30 tokens: ['natural', 'language', 'processing', 'in', 'python', 'authors', 'steven', 'bird', 'ewan', 'klein', 'edward', 'loper', 'v', 'ersion', 'draft', 'only', 'please', 'send', 'feedback', 'to', 'authors', 'copyright', 'the', 'authors', 'license', 'creative', 'commons', 'attributionnoncommercialno', 'derivative', 'works']



In [12]:
stop_words = set(stopwords.words('english'))

stop_words_found = [w for w in tokens if w in stop_words]
valid_words = [w for w in tokens if w not in stop_words and w.strip() != '']

print("STEP 7 — Stop Word Removal")
print(f"  Total Stop Words Found   : {len(stop_words_found):,}")
print(f"  Valid Words After Removal: {len(valid_words):,}")
print(f"  Sample stop words found  : {stop_words_found[:15]}")
print(f"  Sample valid words       : {valid_words[:20]}")
print()

STEP 7 — Stop Word Removal
  Total Stop Words Found   : 45,213
  Valid Words After Removal: 67,905
  Sample stop words found  : ['in', 'only', 'to', 'the', 'to', 'the', 'the', 'of', 'the', 'of', 'and', 'and', 'and', 'the', 'of']
  Sample valid words       : ['natural', 'language', 'processing', 'python', 'authors', 'steven', 'bird', 'ewan', 'klein', 'edward', 'loper', 'v', 'ersion', 'draft', 'please', 'send', 'feedback', 'authors', 'copyright', 'authors']



In [13]:
print("=" * 50)
print("WORD COUNT SUMMARY")
print("=" * 50)
print(f"  Total tokens (raw)         : {len(tokens):,}")
print(f"  Stop words count           : {len(stop_words_found):,}")
print(f"  Valid words (after removal): {len(valid_words):,}")
print("=" * 50)

WORD COUNT SUMMARY
  Total tokens (raw)         : 113,118
  Stop words count           : 45,213
  Valid words (after removal): 67,905


In [14]:
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(word) for word in valid_words]

print("STEP 9 — Stemming (Porter Stemmer)")
print(f"  Total stemmed words: {len(stemmed_words):,}")
print("\n  Original  →  Stemmed (first 20 pairs):")
print("  " + "-" * 40)
for orig, stem in zip(valid_words[:20], stemmed_words[:20]):
    print(f"  {orig:<20} →  {stem}")

STEP 9 — Stemming (Porter Stemmer)
  Total stemmed words: 67,905

  Original  →  Stemmed (first 20 pairs):
  ----------------------------------------
  natural              →  natur
  language             →  languag
  processing           →  process
  python               →  python
  authors              →  author
  steven               →  steven
  bird                 →  bird
  ewan                 →  ewan
  klein                →  klein
  edward               →  edward
  loper                →  loper
  v                    →  v
  ersion               →  ersion
  draft                →  draft
  please               →  pleas
  send                 →  send
  feedback             →  feedback
  authors              →  author
  copyright            →  copyright
  authors              →  author


In [15]:
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(word) for word in valid_words]

print("STEP 10 — Lemmatization (WordNet Lemmatizer)")
print(f"  Total lemmatized words: {len(lemmatized_words):,}")
print("\n  Original  →  Lemmatized (first 20 pairs):")
print("  " + "-" * 40)
for orig, lem in zip(valid_words[:20], lemmatized_words[:20]):
    print(f"  {orig:<20} →  {lem}")

STEP 10 — Lemmatization (WordNet Lemmatizer)
  Total lemmatized words: 67,905

  Original  →  Lemmatized (first 20 pairs):
  ----------------------------------------
  natural              →  natural
  language             →  language
  processing           →  processing
  python               →  python
  authors              →  author
  steven               →  steven
  bird                 →  bird
  ewan                 →  ewan
  klein                →  klein
  edward               →  edward
  loper                →  loper
  v                    →  v
  ersion               →  ersion
  draft                →  draft
  please               →  please
  send                 →  send
  feedback             →  feedback
  authors              →  author
  copyright            →  copyright
  authors              →  author


In [16]:
comparison_df = pd.DataFrame({
    'Original'    : valid_words[:30],
    'Stemmed'     : stemmed_words[:30],
    'Lemmatized'  : lemmatized_words[:30]
})
print("PREPROCESSING COMPARISON TABLE (first 30 words):")
print(comparison_df.to_string(index=True))

PREPROCESSING COMPARISON TABLE (first 30 words):
                      Original                     Stemmed                  Lemmatized
0                      natural                       natur                     natural
1                     language                     languag                    language
2                   processing                     process                  processing
3                       python                      python                      python
4                      authors                      author                      author
5                       steven                      steven                      steven
6                         bird                        bird                        bird
7                         ewan                        ewan                        ewan
8                        klein                       klein                       klein
9                       edward                      edward                      e

---
## Q1(c): Feature Extraction
### Part 1 — One Hot Encoding

In [17]:
unique_words = list(dict.fromkeys(valid_words))[:50]
ohe_matrix = np.zeros((len(unique_words), len(unique_words)), dtype=int)
for i, word in enumerate(unique_words):
    ohe_matrix[i][i] = 1

ohe_df = pd.DataFrame(ohe_matrix, index=unique_words, columns=unique_words)

print("ONE HOT ENCODING OUTPUT")
print(f"Shape: {ohe_df.shape}  (words × categories)")
print()
print(ohe_df.iloc[:10, :10].to_string())
print("\n... (showing 10×10 of full", ohe_df.shape, "matrix)")

ONE HOT ENCODING OUTPUT
Shape: (50, 50)  (words × categories)

            natural  language  processing  python  authors  steven  bird  ewan  klein  edward
natural           1         0           0       0        0       0     0     0      0       0
language          0         1           0       0        0       0     0     0      0       0
processing        0         0           1       0        0       0     0     0      0       0
python            0         0           0       1        0       0     0     0      0       0
authors           0         0           0       0        1       0     0     0      0       0
steven            0         0           0       0        0       1     0     0      0       0
bird              0         0           0       0        0       0     1     0      0       0
ewan              0         0           0       0        0       0     0     1      0       0
klein             0         0           0       0        0       0     0     0      1      

### Part 2 — TF-IDF

In [18]:
# ── TF-IDF
def preprocess_page(text):
    """Apply the same pipeline to a single page."""
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and w.strip()]
    return ' '.join(tokens)

print("Preprocessing pages for TF-IDF corpus...")
corpus = [preprocess_page(p) for p in all_text_pages]
corpus = [c for c in corpus if c.strip()]
print(f"✅ Corpus ready: {len(corpus)} documents (pages)")

Preprocessing pages for TF-IDF corpus...
✅ Corpus ready: 371 documents (pages)


In [19]:

tfidf = TfidfVectorizer(max_features=100)
tfidf_matrix = tfidf.fit_transform(corpus)

feature_names = tfidf.get_feature_names_out()

print("TF-IDF FEATURE NAMES (top 100 words):")
print("-" * 60)
for i, name in enumerate(feature_names):
    print(f"  {i+1:3}. {name}")
    if (i + 1) % 25 == 0:
        print()

TF-IDF FEATURE NAMES (top 100 words):
------------------------------------------------------------
    1. also
    2. bird
    3. called
    4. case
    5. chapter
    6. characters
    7. chart
    8. chunk
    9. code
   10. context
   11. corpus
   12. data
   13. different
   14. dog
   15. draft
   16. edge
   17. eg
   18. example
   19. expression
   20. expressions
   21. feature
   22. following
   23. function
   24. grammar
   25. however

   26. information
   27. input
   28. introduction
   29. january
   30. klein
   31. language
   32. like
   33. line
   34. linguistic
   35. list
   36. listing
   37. loper
   38. many
   39. may
   40. natural
   41. nd
   42. need
   43. new
   44. nltk
   45. nn
   46. nns
   47. noun
   48. np
   49. number
   50. often

   51. one
   52. order
   53. parse
   54. parser
   55. parsing
   56. phrase
   57. pp
   58. print
   59. processing
   60. program
   61. programming
   62. python
   63. regular
   64. return
   65. rst
   6

In [20]:

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"Page_{i+1}" for i in range(len(corpus))]
)

print("TF-IDF VALUES TABLE (first 10 pages × first 15 features):")
print(tfidf_df.iloc[:10, :15].round(4).to_string())

TF-IDF VALUES TABLE (first 10 pages × first 15 features):
         also    bird  called    case  chapter  characters   chart   chunk  code  context  corpus    data  different  dog   draft
Page_1    0.0  0.0000     0.0  0.0000      0.0      0.0000  0.0000  0.0000   0.0   0.0000  0.0000  0.0000     0.0000  0.0  0.0000
Page_2    0.0  0.5035     0.0  0.0000      0.0      0.0000  0.0000  0.0000   0.0   0.0000  0.0000  0.0000     0.0000  0.0  0.4151
Page_3    0.0  0.0000     0.0  0.0000      0.0      0.1963  0.0000  0.0000   0.0   0.0000  0.0000  0.0000     0.0000  0.0  0.0000
Page_4    0.0  0.0482     0.0  0.0000      0.0      0.0000  0.0000  0.0000   0.0   0.0000  0.0000  0.3143     0.1138  0.0  0.0000
Page_5    0.0  0.0704     0.0  0.0000      0.0      0.0000  0.0000  0.0000   0.0   0.0000  0.0000  0.0000     0.0000  0.0  0.1160
Page_6    0.0  0.0765     0.0  0.0000      0.0      0.0000  0.0000  0.0000   0.0   0.0000  0.1780  0.0000     0.1806  0.0  0.0000
Page_7    0.0  0.0431     0.0  0

In [22]:
mean_tfidf = tfidf_df.mean(axis=0).sort_values(ascending=False)
mean_tfidf_df = mean_tfidf.reset_index()
mean_tfidf_df.columns = ['Word', 'Mean_TF-IDF']

print("TOP 20 WORDS BY MEAN TF-IDF SCORE:")
print(mean_tfidf_df.head(20).to_string(index=False))

TOP 20 WORDS BY MEAN TF-IDF SCORE:
      Word  Mean_TF-IDF
  language     0.088022
      word     0.079583
     words     0.078095
        np     0.074745
      bird     0.066929
     klein     0.065644
     loper     0.064876
   grammar     0.063026
      text     0.060447
   january     0.060211
   natural     0.060158
    python     0.058255
    string     0.052767
processing     0.052592
      data     0.052339
     print     0.049981
      list     0.049767
     using     0.048996
  function     0.047767
   example     0.047145


---
## Q1(d): TF-IDF Visualizations Using Plotly

> **Note:** Only Plotly is used for all visualizations as required by the assignment.  
> **More than one graph is produced** as instructed.

In [23]:
top50 = mean_tfidf_df.head(50)

fig1 = px.scatter(
    top50,
    x='Word',
    y='Mean_TF-IDF',
    text='Word',
    size='Mean_TF-IDF',
    color='Mean_TF-IDF',
    color_continuous_scale='Viridis',
    title='Graph 1: TF-IDF Scatter Plot — Top 50 Words vs Mean TF-IDF Score',
    labels={
        'Word'       : 'Words',
        'Mean_TF-IDF': 'Mean TF-IDF Score'
    }
)
fig1.update_traces(textposition='top center', marker=dict(opacity=0.8))
fig1.update_layout(
    xaxis_tickangle=-45,
    height=600,
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=11)
)
fig1.show()
print("✅ Graph 1: Scatter Plot rendered.")

✅ Graph 1: Scatter Plot rendered.


In [24]:
top30 = mean_tfidf_df.head(30)

fig2 = px.bar(
    top30,
    x='Word',
    y='Mean_TF-IDF',
    color='Mean_TF-IDF',
    color_continuous_scale='Blues',
    title='Graph 2: TF-IDF Bar Chart — Top 30 Most Important Words',
    labels={
        'Word'       : 'Words',
        'Mean_TF-IDF': 'Mean TF-IDF Score'
    },
    text='Mean_TF-IDF'
)
fig2.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig2.update_layout(
    xaxis_tickangle=-45,
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=11)
)
fig2.show()
print("✅ Graph 2: Bar Chart rendered.")

✅ Graph 2: Bar Chart rendered.


In [25]:

top20 = mean_tfidf_df.head(20).sort_values('Mean_TF-IDF', ascending=True)

fig3 = px.bar(
    top20,
    x='Mean_TF-IDF',
    y='Word',
    orientation='h',
    color='Mean_TF-IDF',
    color_continuous_scale='Oranges',
    title='Graph 3: TF-IDF Horizontal Bar — Top 20 Words Ranked by Importance',
    labels={
        'Word'       : 'Words',
        'Mean_TF-IDF': 'Mean TF-IDF Score'
    },
    text='Mean_TF-IDF'
)
fig3.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig3.update_layout(
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12)
)
fig3.show()
print("✅ Graph 3: Horizontal Bar Chart rendered.")

✅ Graph 3: Horizontal Bar Chart rendered.


In [26]:
top40 = mean_tfidf_df.head(40).copy()
top40['Rank'] = range(1, len(top40) + 1)

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=top40['Rank'],
    y=top40['Mean_TF-IDF'],
    mode='markers+text',
    text=top40['Word'],
    textposition='top center',
    marker=dict(
        size=top40['Mean_TF-IDF'] * 1000,   # scale bubble size
        color=top40['Mean_TF-IDF'],
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title='TF-IDF Score'),
        opacity=0.8
    )
))
fig4.update_layout(
    title='Graph 4: TF-IDF Bubble Scatter — Word Rank vs Score (Top 40 Words)',
    xaxis_title='Word Rank (1 = Most Important)',
    yaxis_title='Mean TF-IDF Score',
    height=600,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=11)
)
fig4.show()
print("✅ Graph 4: Bubble Scatter Plot rendered.")

✅ Graph 4: Bubble Scatter Plot rendered.


In [27]:
top20_words = mean_tfidf_df.head(20)['Word'].tolist()
heatmap_data = tfidf_df[top20_words].head(10)

fig5 = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=top20_words,
    y=heatmap_data.index.tolist(),
    colorscale='RdYlGn',
    colorbar=dict(title='TF-IDF Score')
))
fig5.update_layout(
    title='Graph 5: TF-IDF Heatmap — Top 20 Words Across First 10 Pages',
    xaxis_title='Words',
    yaxis_title='Document Pages',
    height=500,
    xaxis_tickangle=-45,
    font=dict(size=11)
)
fig5.show()
print("✅ Graph 5: TF-IDF Heatmap rendered.")

✅ Graph 5: TF-IDF Heatmap rendered.


---
## Summary of All Steps Completed

| Step | Task | Status |
|------|------|--------|
| Q1(a)-1 | PDF file selected (≥100 pages) | ✅ |
| Q1(a)-2 | PDF read using Python (pypdf) | ✅ |
| Q1(a)-3 | Text extracted from all pages | ✅ |
| Q1(a)-4 | Total number of pages shown | ✅ |
| Q1(a)-5 | Sample extracted text shown | ✅ |
| Q1(b)-1 | Converted to lowercase | ✅ |
| Q1(b)-2 | Removed numbers (Regex: `\d+`) | ✅ |
| Q1(b)-3 | Removed special symbols (Regex: `[^a-z\s]`) | ✅ |
| Q1(b)-4 | Removed extra spaces (Regex: `\s+`) | ✅ |
| Q1(b)-5 | Removed punctuation | ✅ |
| Q1(b)-6 | Tokenized text into words | ✅ |
| Q1(b)-7 | Applied stop word removal | ✅ |
| Q1(b)-8 | Showed stop word count | ✅ |
| Q1(b)-9 | Showed valid word count | ✅ |
| Q1(b)-10 | Applied stemming (Porter Stemmer) | ✅ |
| Q1(b)-11 | Applied lemmatization (WordNet) | ✅ |
| Q1(c)-1 | One Hot Encoding applied & shown in table | ✅ |
| Q1(c)-2 | TF-IDF applied, feature names shown, values in table | ✅ |
| Q1(d) | **5 Plotly graphs** (Scatter, Bar, H-Bar, Bubble, Heatmap) | ✅ |